# 06 — Full Scene Prediction & Spectral Analysis
Apply the trained CNN to the entire orbital strip, visualize the classified map,
and validate class name assignments using continuum-removal band depth analysis.

## Full Scene Prediction

In [ ]:
print("Reshaping data for prediction...")
full_data_cnn = reduced_data.reshape(-1, 10, 1, 1)

print("Generating predictions...")
predicted_probabilities = cnn_model.predict(full_data_cnn)
predicted_labels        = np.argmax(predicted_probabilities, axis=1)
classified_image        = predicted_labels.reshape(hyperspectral_data.shape[0],
                                                    hyperspectral_data.shape[1])
print("Done.")

## Classified Lunar Surface Map

In [ ]:
from matplotlib.colors import ListedColormap

class_names  = ['Mare Basalt', 'Highland Anorthosite', 'Impact Melt',
                 'Pyroclastic Deposit', 'Mixed Terrain']
class_colors = ['#f7e8aa', '#934b43', '#708090', '#afafaf', '#3a4e48']
lunar_cmap   = ListedColormap(class_colors)

plt.figure(figsize=(14, 10))
plt.imshow(classified_image, cmap=lunar_cmap)
plt.title("CNN-Classified Lunar Surface — Chandrayaan-2 IIRS", fontsize=16)
cbar = plt.colorbar(ticks=range(5))
cbar.set_label("Spectral Class", fontsize=14)
cbar.set_ticklabels(class_names)
plt.xlabel("Pixel", fontsize=14)
plt.ylabel("Scan Line", fontsize=14)
save_fig("cnn_classified_lunar_surface")
plt.show()

## Class Distribution

In [ ]:
class_counts      = np.bincount(predicted_labels, minlength=5)
class_percentages = class_counts / len(predicted_labels) * 100
class_df = pd.DataFrame({'Class': class_names, 'Count': class_counts,
                          'Percentage': class_percentages})
print(class_df.to_string(index=False))

plt.figure(figsize=(12, 7))
bars = plt.bar(class_names, class_percentages, color=class_colors)
plt.title("Distribution of Spectral Classes", fontsize=16)
plt.xlabel("Lunar Surface Class", fontsize=14)
plt.ylabel("Percentage of Surface Area (%)", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.grid(True, axis='y', alpha=0.3)
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., h + 0.5,
             f'{h:.1f}%', ha='center', va='bottom', fontsize=12)
plt.tight_layout()
save_fig("class_distribution")
plt.show()

## Spectral Signatures with IIRS Wavelength Axis
IIRS wavelength range: 800 nm → 5000 nm across 256 bands.  
Shaded regions mark the diagnostic 1µm and 2µm absorption windows.

In [ ]:
class_spectra = []
for i in range(5):
    mask = (predicted_labels == i)
    avg  = np.mean(flattened_data[mask], axis=0) if np.any(mask) else np.zeros(flattened_data.shape[1])
    class_spectra.append(avg)

wavelengths = np.linspace(800, 5000, hyperspectral_data.shape[2])
print(f"Wavelength range: {wavelengths[0]:.1f} nm  →  {wavelengths[-1]:.1f} nm")

plt.figure(figsize=(16, 10))
for i, spectrum in enumerate(class_spectra):
    plt.plot(wavelengths, spectrum, label=class_names[i],
             linewidth=2.5, color=class_colors[i])
plt.axvspan(900,  1100, alpha=0.08, color='red')
plt.axvline(1000, color='red',  linestyle='--', linewidth=1, label='1µm band (pyroxene/olivine)')
plt.axvspan(1900, 2100, alpha=0.08, color='blue')
plt.axvline(2000, color='blue', linestyle='--', linewidth=1, label='2µm band (pyroxene type)')
plt.title("Average Spectral Signatures — IIRS Chandrayaan-2", fontsize=16)
plt.xlabel("Wavelength (nm)", fontsize=14)
plt.ylabel("Normalized Reflectance", fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
save_fig("spectral_signatures_with_wavelengths")
plt.show()

## Band Depth Analysis — Mineral Validation
Continuum-removal band depths at 1µm and 2µm validate the assigned class names
against known lunar mineral spectral signatures (pyroxene, olivine, anorthosite).

In [ ]:
def band_at(target_nm):
    return int(np.argmin(np.abs(wavelengths - target_nm)))

b_800  = band_at(800);   b_1000 = band_at(1000);  b_1300 = band_at(1300)
b_1600 = band_at(1600);  b_2000 = band_at(2000);  b_2500 = band_at(2500)

results = []
print(f"{'Class':<25} {'BD_1µm':>8} {'BD_2µm':>8} {'Slope':>10}  Interpretation")
print("-" * 85)

for name, spectrum in zip(class_names, class_spectra):
    c1 = np.interp(wavelengths[b_1000],
                   [wavelengths[b_800],  wavelengths[b_1300]],
                   [spectrum[b_800],     spectrum[b_1300]])
    bd_1um = 1.0 - (spectrum[b_1000] / c1) if c1 > 0 else 0.0

    c2 = np.interp(wavelengths[b_2000],
                   [wavelengths[b_1600], wavelengths[b_2500]],
                   [spectrum[b_1600],    spectrum[b_2500]])
    bd_2um = 1.0 - (spectrum[b_2000] / c2) if c2 > 0 else 0.0

    slope = ((spectrum[b_1300] - spectrum[b_800]) /
             (wavelengths[b_1300] - wavelengths[b_800]))

    if   bd_1um > 0.05 and bd_2um > 0.05: note = "✓ Mafic (pyroxene) — Mare Basalt / Impact Melt"
    elif bd_1um > 0.05 and bd_2um < 0.02: note = "✓ Olivine-dominated — Pyroclastic / Impact Melt"
    elif bd_1um < 0.02 and slope > 0:     note = "✓ Featureless + red slope — Highland Anorthosite"
    elif bd_1um < 0.02 and slope < 0:     note = "✓ Featureless + flat — Mixed Terrain"
    else:                                 note = "⚠ Ambiguous — review manually"

    results.append({'Class': name, 'BD_1um': bd_1um, 'BD_2um': bd_2um, 'Slope': slope})
    print(f"{name:<25} {bd_1um:>8.4f} {bd_2um:>8.4f} {slope:>10.6f}  {note}")

print("\n── Reference ────────────────────────────────────────────────────")
print("BD_1µm > 0.05  →  mafic absorption (pyroxene / olivine)")
print("BD_2µm > 0.05  →  pyroxene confirmed (LCP/HCP)")
print("BD_1µm < 0.02  →  plagioclase-rich / anorthosite / flat terrain")
print("Slope  > 0     →  red slope = mature highland regolith")

In [ ]:
results_df = pd.DataFrame(results)
x_pos = np.arange(len(class_names))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x_pos - width/2, results_df['BD_1um'], width, label='Band Depth @ 1µm', color='tomato')
ax.bar(x_pos + width/2, results_df['BD_2um'], width, label='Band Depth @ 2µm', color='steelblue')
ax.set_xlabel('Spectral Class', fontsize=13)
ax.set_ylabel('Band Depth', fontsize=13)
ax.set_title('Absorption Band Depths per Class — Mineral Validation', fontsize=15)
ax.set_xticks(x_pos)
ax.set_xticklabels(class_names, rotation=30, ha='right')
ax.legend(fontsize=12)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
save_fig("band_depth_validation")
plt.show()